# 6 — Optional Cleanup

⚠️ **Permanently deletes AWS resources. Read each cell before running.**

This notebook tears down everything created by notebooks 1–5 in the **safe order** the AWS APIs require:

| Step | What it removes | Owned by |
|------|-----------------|----------|
| 0 | Model Package Group (includes stale baseline.json) | Training pipeline (out-of-band) |
| 1 | SageMaker endpoint, endpoint config, model | Notebook 2 (out-of-band) |
| 2 | Drift-monitor Lambda, EventBridge rule, SNS topic, CloudWatch dashboard + alarms | Notebook 3 (out-of-band) |
| 3 | CloudFormation stack — SageMaker domain, monitoring-writer Lambda, IAM roles, SQS, S3 buckets, Athena DB | CFN |

**VPC/Subnet Preservation**: By default, this notebook preserves VPC and subnets for reuse. Set `DELETE_VPC=True` in the configuration cell to delete them.

Both shell scripts default to **dry-run** — they list what would be deleted but do nothing. Switch to `--execute` only when the dry-run output looks right.

[nbviewer](https://nbviewer.org/github/aws-samples/mlops-best-practices-on-aws/blob/main/sagemaker-automated-drift-and-trend-monitoring/notebooks/6_optional_cleanup.ipynb)

## Setup & Configuration

In [1]:
import sys
import os
import subprocess
from pathlib import Path
import boto3

# Find project root
project_root = Path.cwd()
while not (project_root / '.env').exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / '.env')

from src.config.config import (
    AWS_DEFAULT_REGION,
    ENDPOINT_NAME,
    MLFLOW_MODEL_NAME,
)

REGION = AWS_DEFAULT_REGION
# Model Package Group name matches MLflow model name in this project
MODEL_PACKAGE_GROUP = MLFLOW_MODEL_NAME

# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION: What to delete
# ═══════════════════════════════════════════════════════════════════════════

# Set to True to delete VPC, subnets, security groups, route tables, etc.
# Set to False (default) to preserve VPC infrastructure for reuse
DELETE_VPC = False

# Set to True to also delete ECR container images
DELETE_ECR = False

print('═' * 72)
print('CLEANUP CONFIGURATION')
print('═' * 72)
print(f'Region:                  {REGION}')
print(f'Endpoint name:           {ENDPOINT_NAME}')
print(f'Model package group:     {MODEL_PACKAGE_GROUP}')
print(f'Project root:            {project_root}')
print()
print(f'Delete VPC/Subnets:      {DELETE_VPC}')
print(f'Delete ECR images:       {DELETE_ECR}')
print('═' * 72)

════════════════════════════════════════════════════════════════════════
CLEANUP CONFIGURATION
════════════════════════════════════════════════════════════════════════
Region:                  us-west-2
Endpoint name:           fraud-detector-endpoint
Model package group:     fraud-detection
Project root:            /Users/skoppar/workspace/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring

Delete VPC/Subnets:      False
Delete ECR images:       False
════════════════════════════════════════════════════════════════════════


---
## Step 0 — Model Package Group (contains stale baseline.json)

The Model Package Group stores all model versions and their `baseline.json` metadata. If you're getting "Iceberg snapshot does not exist" errors, this is where the stale snapshot IDs live.

**Important**: Delete this BEFORE retraining so the new training run creates a fresh model package group.

### 0.1 — Preview model packages

In [2]:
sm = boto3.client('sagemaker', region_name=REGION)

def safe(call, label):
    try:
        return call()
    except Exception as e:
        print(f'  (skip {label}: {e.__class__.__name__})')
        return None

print(f'Model Package Group: {MODEL_PACKAGE_GROUP}')
print('=' * 72)

# Check if group exists
try:
    group_info = sm.describe_model_package_group(ModelPackageGroupName=MODEL_PACKAGE_GROUP)
    print(f'✓ Group exists')
    print(f'  Created: {group_info["CreationTime"]}')
    print(f'  Status:  {group_info["ModelPackageGroupStatus"]}')
    
    # List all packages in the group
    packages = sm.list_model_packages(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP,
        SortBy='CreationTime',
        SortOrder='Descending',
        MaxResults=10
    )
    
    if packages['ModelPackageSummaryList']:
        print(f'\nModel Packages ({len(packages["ModelPackageSummaryList"])} shown, may be more):')
        for pkg in packages['ModelPackageSummaryList']:
            version = pkg['ModelPackageArn'].split('/')[-1]
            status = pkg['ModelApprovalStatus']
            created = pkg['CreationTime']
            print(f'  - Version {version:3s}  {status:20s}  {created}')
    else:
        print('\nNo model packages found in this group.')
        
except sm.exceptions.ResourceNotFound:
    print('⚠ Model package group does not exist (nothing to delete)')
except Exception as e:
    print(f'❌ Error: {e}')

Model Package Group: fraud-detection
✓ Group exists
  Created: 2026-06-24 22:13:35.440000-07:00
  Status:  Completed

No model packages found in this group.


### 0.2 — Delete model package group (uncomment to execute)

This deletes ALL model versions and their metadata (including the stale baseline.json with old Iceberg snapshot IDs).

In [ ]:
try:
    # First, delete all model packages in the group
    all_packages = []
    next_token = None
    
    while True:
        kwargs = {'ModelPackageGroupName': MODEL_PACKAGE_GROUP, 'MaxResults': 100}
        if next_token:
            kwargs['NextToken'] = next_token
        
        response = sm.list_model_packages(**kwargs)
        all_packages.extend(response['ModelPackageSummaryList'])
        
        if 'NextToken' not in response:
            break
        next_token = response['NextToken']
    
    print(f'Deleting {len(all_packages)} model package(s)...')
    for pkg in all_packages:
        arn = pkg['ModelPackageArn']
        version = arn.split('/')[-1]
        sm.delete_model_package(ModelPackageName=arn)
        print(f'  ✓ Deleted version {version}')
    
    # Then delete the group itself
    sm.delete_model_package_group(ModelPackageGroupName=MODEL_PACKAGE_GROUP)
    print(f'\n✓ Deleted model package group: {MODEL_PACKAGE_GROUP}')
    
except sm.exceptions.ResourceNotFound:
    print(f'⚠ Model package group {MODEL_PACKAGE_GROUP} does not exist')
except Exception as e:
    print(f'❌ Error: {e}')

Deleting 0 model package(s)...

✓ Deleted model package group: fraud-detection


---
## Step 1 — SageMaker endpoint + config + model

Notebook 2 creates these via `boto3` (outside CFN), so CFN won't clean them up. We do this first because endpoints are billed per-hour for instance time.

The cell below **only lists** matching resources. To delete, uncomment the marked lines.

In [9]:
def safe(call, label):
    try:
        return call()
    except Exception as e:
        print(f'  (skip {label}: {e.__class__.__name__})')
        return None

print('Endpoint:')
ep = safe(lambda: sm.describe_endpoint(EndpointName=ENDPOINT_NAME), 'endpoint')
if ep:
    print(f'  - {ENDPOINT_NAME}  status={ep["EndpointStatus"]}  config={ep["EndpointConfigName"]}')
else:
    print(f'  (no endpoint named {ENDPOINT_NAME})')

print('\nEndpoint configs (matching "fraud"):')
configs = sm.list_endpoint_configs(MaxResults=100).get('EndpointConfigs', [])
fraud_configs = [c['EndpointConfigName'] for c in configs if 'fraud' in c['EndpointConfigName'].lower()]
for c in fraud_configs:
    print(f'  - {c}')
if not fraud_configs:
    print('  (none)')

print('\nModels (matching "fraud"):')
models = sm.list_models(MaxResults=100).get('Models', [])
fraud_models = [m['ModelName'] for m in models if 'fraud' in m['ModelName'].lower()]
for m in fraud_models:
    print(f'  - {m}')
if not fraud_models:
    print('  (none)')

# ── To delete: uncomment the block below ─────────────────────────────────────
if ep:
    print(f'\nDeleting endpoint: {ENDPOINT_NAME}')
    sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print(f'✓ Deleted endpoint {ENDPOINT_NAME}')

for c in fraud_configs:
    sm.delete_endpoint_config(EndpointConfigName=c)
    print(f'✓ Deleted endpoint config {c}')

for m in fraud_models:
    sm.delete_model(ModelName=m)
    print(f'✓ Deleted model {m}')

Endpoint:
  - fraud-detector-endpoint  status=InService  config=fraud-detector-endpoint-config-1783635382

Endpoint configs (matching "fraud"):
  - fraud-detector-endpoint-config-1783635382
  - fraud-detector-endpoint-config-1783635314
  - fraud-detector-endpoint-config-1782949797
  - fraud-detector-endpoint-config-1782875115
  - fraud-detector-endpoint-config-1782512938
  - fraud-detector-endpoint-config-1782506047
  - fraud-detector-endpoint-config-1782501307
  - fraud-detector-endpoint-config-1782479893
  - fraud-detector-endpoint-config-1782453815
  - fraud-detector-endpoint-config-1782451316
  - fraud-detector-endpoint-config-1782450254
  - fraud-detector-endpoint-config-1782442557
  - fraud-detector-endpoint-config-1782428291
  - fraud-detector-endpoint-config-1782406505
  - fraud-detector-endpoint-config-1782364417-fixed
  - fraud-detector-endpoint-config-1782364417

Models (matching "fraud"):
  - fraud-detector-endpoint-model-1783635381
  - fraud-detector-endpoint-model-1783635

---
## Step 2 — Out-of-band drift-monitor resources

`scripts/delete_infrastructure.sh` tears down what notebook 3 deploys outside CFN: drift-monitor Lambda, EventBridge schedule, SNS topic + subscriptions, CloudWatch dashboard + 5 alarms.

### 2.1 — Dry-run preview (always safe)

In [10]:
subprocess.run(['bash', 'scripts/delete_infrastructure.sh'], cwd=project_root)

╔════════════════════════════════════════════════════════════════════╗
║  Out-of-band drift-monitor resource cleanup                        ║
║  DRY-RUN (no changes — pass --execute to actually delete)
╚════════════════════════════════════════════════════════════════════╝

  Region:           us-west-2
  EventBridge rule: fraud-detection-drift-check
  Lambda function:  fraud-detection-drift-monitor
  SNS topic:        fraud-detection-drift-alerts
  Dashboard:        FraudDetection-DriftMonitoring
  CW alarms:        5 alarms
  ECR repo:         fraud-detection-drift-monitor  (delete = false)

[1/5] EventBridge rule...
  [DRY-RUN] would run: aws events remove-targets --rule fraud-detection-drift-check --ids 1 --region us-west-2
  [DRY-RUN] would run: aws events delete-rule --name fraud-detection-drift-check --region us-west-2
  ✓ Rule: fraud-detection-drift-check

[2/5] Lambda function...
  [DRY-RUN] would run: aws lambda delete-function --function-name fraud-detection-drift-monitor --r

CompletedProcess(args=['bash', 'scripts/delete_infrastructure.sh'], returncode=0)

### 2.2 — Execute (uncomment to actually delete)

Pass `--ecr` if you also want to delete the ECR repo (otherwise the image is kept around for fast redeploy).

In [12]:
# ── Uncomment to execute deletion ───────────────────────────────────────────
# ecr_flag = ['--ecr'] if DELETE_ECR else []
# subprocess.run(['bash', 'scripts/delete_infrastructure.sh', '--execute'] + ecr_flag, cwd=project_root)

---
## Step 3 — CloudFormation stack

`cloudformation/delete-main-stack.sh` drains things CFN can't drain itself (Athena Iceberg tables, S3 buckets, running JupyterLab apps, Lake Formation grants) then issues `delete-stack` and polls until done.

**VPC Preservation**: By default, VPC resources (VPC, subnets, security groups, route tables, IGW) are PRESERVED for reuse. Set `DELETE_VPC=True` in the configuration cell above to delete them.

It **refuses to run** in execute mode if the drift-monitor Lambda is still present — that's the most common deletion failure, because the Lambda holds refs to the CFN-owned SQS queue, SNS topic, and IAM role.

### 3.1 — Dry-run preview

In [13]:
vpc_flag = ['--delete-vpc'] if DELETE_VPC else ['--preserve-vpc']
subprocess.run(['bash', 'cloudformation/delete-main-stack.sh'] + vpc_flag, cwd=project_root)

╔════════════════════════════════════════════════════════════════════╗
║  CloudFormation stack delete                                       ║
║  DRY-RUN (no changes — pass --execute to actually delete)
╚════════════════════════════════════════════════════════════════════╝

  Stack:       fraud-detection-monitoring
  Region:      us-west-2
  VPC mode:    PRESERVE VPC/subnets (pass --delete-vpc to delete)

  Current status: UPDATE_COMPLETE

  ⚠️  Drift-monitor Lambda still exists.
  In execute mode this script would refuse to run.
  Resolve by running: scripts/delete_infrastructure.sh --execute

[1/6] Dropping Glue/Athena tables in 'fraud_detection'...
  [DRY-RUN] would drop fraud_detection.drifted_data
  [DRY-RUN] would drop fraud_detection.evaluation_data
  [DRY-RUN] would drop fraud_detection.ground_truth
  [DRY-RUN] would drop fraud_detection.ground_truth_updates
  [DRY-RUN] would drop fraud_detection.inference_responses
  [DRY-RUN] would drop fraud_detection.monitoring_responses
  [

CompletedProcess(args=['bash', 'cloudformation/delete-main-stack.sh', '--preserve-vpc'], returncode=0)

### 3.2 — Execute (uncomment to actually delete)

Defaults to stack name from config. VPC preservation is controlled by `DELETE_VPC` variable in the configuration cell.

In [15]:
# ── Uncomment to execute deletion ───────────────────────────────────────────
vpc_flag = ['--delete-vpc'] if DELETE_VPC else ['--preserve-vpc']
subprocess.run(['bash', 'cloudformation/delete-main-stack.sh', '--execute'] + vpc_flag, cwd=project_root)

╔════════════════════════════════════════════════════════════════════╗
║  CloudFormation stack delete                                       ║
║  EXECUTE (resources WILL be deleted)
╚════════════════════════════════════════════════════════════════════╝

  Stack:       fraud-detection-monitoring
  Region:      us-west-2
  VPC mode:    PRESERVE VPC/subnets (pass --delete-vpc to delete)

  Current status: UPDATE_COMPLETE

  ⚠️  Drift-monitor Lambda still exists.
  Run scripts/delete_infrastructure.sh --execute first to drain the
  out-of-band resources, then re-run this script.


CompletedProcess(args=['bash', 'cloudformation/delete-main-stack.sh', '--execute', '--preserve-vpc'], returncode=1)

---
## Step 4 — What's left after delete-stack?

CFN deletes everything inside the stack but leaves these behind by default:

- **VPC resources** (if `DELETE_VPC=False`) — VPC, subnets, security groups, route tables, Internet Gateway. These are preserved for faster redeployment. To delete them, set `DELETE_VPC=True` and re-run Step 3.

- **CloudWatch log groups** — `/aws/lambda/*`, `/aws/sagemaker/*`. AWS keeps them so you can debug failed deletes. Delete manually if you want a fully clean account:
  ```bash
  aws logs describe-log-groups --log-group-name-prefix /aws/lambda/fraud --region us-west-2 \
      --query 'logGroups[].logGroupName' --output text | \
      xargs -n1 aws logs delete-log-group --region us-west-2 --log-group-name
  ```

- **ECR drift-monitor image** (if `DELETE_ECR=False`) — kept by default in Step 2 for faster redeployment. Set `DELETE_ECR=True` and re-run Step 2.2 to delete.

- **Model Package Group** (if you skipped Step 0) — contains stale baseline.json with old Iceberg snapshot IDs. Delete via Step 0.2 or:
  ```bash
  aws sagemaker delete-model-package-group --model-package-group-name fraud-detection
  ```

---

## Summary of Cleanup Order

For a **complete** clean slate (deletes everything):

1. **Step 0.2**: Delete model package group (stale baseline.json)
2. **Step 1**: Delete SageMaker endpoint + configs + models  
3. **Step 2.2**: Delete drift-monitor Lambda + SNS + EventBridge + CloudWatch
4. Set `DELETE_VPC=True` and `DELETE_ECR=True` in config cell
5. **Step 3.2**: Delete CloudFormation stack (includes VPC)

For a **partial** cleanup (preserves VPC/subnets for reuse):

1. **Step 0.2**: Delete model package group
2. **Step 1**: Delete SageMaker endpoint + configs + models
3. **Step 2.2**: Delete drift-monitor Lambda resources
4. Keep `DELETE_VPC=False` (default)
5. **Step 3.2**: Delete CloudFormation stack (preserves VPC)

After cleanup, run notebooks 1-3 again to redeploy with fresh state.